# Crop Recommendation System – Machine Learning

This Google Colab notebook trains and evaluates machine-learning models using the uploaded `standardized_crops.csv` dataset.

**Workflow:** Load data → inspect → clean → visualize → prepare features → train models → compare accuracy → confusion matrix → classification report → feature importance → test prediction.


In [ ]:
# CELL 1 — Import Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

print("Libraries imported successfully!")


In [ ]:
# CELL 2 — Upload and Load the CSV File

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("File loaded:", file_name)
print("Dataset shape:", df.shape)

display(df.head())


In [ ]:
# CELL 3 — Inspect the Dataset

print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())


In [ ]:
# CELL 4 — Basic Statistical Summary

display(df.describe(include="all"))


In [ ]:
# CELL 5 — Remove Duplicate Rows

before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Rows before removing duplicates:", before)
print("Rows after removing duplicates:", after)
print("Duplicates removed:", before - after)


In [ ]:
# CELL 6 — Handle Missing Values

# Numerical columns: fill missing values with median
numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Categorical columns: fill missing values with mode
categorical_columns = df.select_dtypes(include="object").columns

for column in categorical_columns:
    if not df[column].mode().empty:
        df[column] = df[column].fillna(df[column].mode()[0])

print("Missing values after cleaning:")
print(df.isnull().sum())


In [ ]:
# CELL 7 — Identify the Target Column

print("Columns available in the dataset:")
for i, column in enumerate(df.columns):
    print(i, ":", column)

# The code tries to detect the crop target automatically.
possible_targets = [
    "label", "Label",
    "crop", "Crop",
    "Crop Name", "CropName",
    "target", "Target"
]

target_column = next(
    (col for col in possible_targets if col in df.columns),
    df.columns[-1]
)

print("\nSelected target column:", target_column)


In [ ]:
# CELL 8 — Explore the Target Classes

print("Number of crop classes:", df[target_column].nunique())
print("\nCrop classes:")
print(df[target_column].unique())

print("\nClass distribution:")
display(df[target_column].value_counts())


In [ ]:
# CELL 9 — Visualize Crop Distribution

plt.figure(figsize=(12, 6))

order = df[target_column].value_counts().index

sns.countplot(
    data=df,
    x=target_column,
    order=order
)

plt.title("Crop Class Distribution")
plt.xlabel("Crop")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# CELL 10 — Correlation Heatmap for Numerical Features

numeric_df = df.select_dtypes(include=np.number)

if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(10, 7))
    sns.heatmap(
        numeric_df.corr(),
        annot=True,
        cmap="coolwarm",
        fmt=".2f"
    )
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns for a correlation heatmap.")


In [ ]:
# CELL 11 — Separate Features and Target

X = df.drop(columns=[target_column])
y = df[target_column].copy()

print("Feature columns:")
print(X.columns.tolist())

print("\nTarget column:", target_column)
print("\nFeature shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
# CELL 12 — Encode Categorical Features

# Save encoders so that new input values can be encoded later.
feature_encoders = {}

for column in X.select_dtypes(include="object").columns:
    encoder = LabelEncoder()
    X[column] = encoder.fit_transform(X[column].astype(str))
    feature_encoders[column] = encoder

# Encode target if it is categorical
target_encoder = None

if y.dtype == "object":
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y.astype(str))

print("Categorical feature encoding completed.")
print("Encoded feature columns:", list(feature_encoders.keys()))


In [ ]:
# CELL 13 — Check the Prepared Data

display(X.head())

print("X data types:")
print(X.dtypes)

print("\nFirst target values:")
print(y[:10])


In [ ]:
# CELL 14 — Train/Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:
# CELL 15 — Train Decision Tree

dt_model = DecisionTreeClassifier(
    random_state=42
)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, dt_pred)

print("Decision Tree Accuracy:", round(dt_accuracy * 100, 2), "%")


In [ ]:
# CELL 16 — Train Random Forest

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", round(rf_accuracy * 100, 2), "%")


In [ ]:
# CELL 17 — Train K-Nearest Neighbors

knn_model = KNeighborsClassifier(
    n_neighbors=5
)

knn_model.fit(X_train, y_train)

knn_pred = knn_model.predict(X_test)

knn_accuracy = accuracy_score(y_test, knn_pred)

print("KNN Accuracy:", round(knn_accuracy * 100, 2), "%")


In [ ]:
# CELL 18 — Compare Model Accuracy

results = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Random Forest",
        "KNN"
    ],
    "Accuracy": [
        dt_accuracy,
        rf_accuracy,
        knn_accuracy
    ]
})

results["Accuracy (%)"] = results["Accuracy"] * 100

display(results)


In [ ]:
# CELL 19 — Accuracy Comparison Graph

plt.figure(figsize=(9, 5))

sns.barplot(
    data=results,
    x="Model",
    y="Accuracy (%)"
)

plt.ylim(0, 100)
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy (%)")
plt.xlabel("Model")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 20 — Random Forest Classification Report

if target_encoder is not None:
    class_names = target_encoder.classes_
else:
    class_names = [str(c) for c in sorted(np.unique(y))]

print("Random Forest Classification Report:")
print(
    classification_report(
        y_test,
        rf_pred,
        target_names=class_names
    )
)


In [ ]:
# CELL 21 — Random Forest Confusion Matrix

cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(12, 9))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Crop")
plt.ylabel("Actual Crop")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# CELL 22 — Random Forest Feature Importance

importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

display(importance_df)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=importance_df,
    x="Importance",
    y="Feature"
)

plt.title("Random Forest Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
# CELL 23 — Prediction Function

def predict_crop(input_values):
    '''
    input_values must be a dictionary whose keys are the
    original feature-column names.
    '''

    input_df = pd.DataFrame([input_values])

    # Encode categorical values using the encoders fitted above
    for column, encoder in feature_encoders.items():
        value = str(input_df.loc[0, column])

        if value not in encoder.classes_:
            raise ValueError(
                f"Unknown value '{value}' for column '{column}'. "
                f"Allowed values include: {list(encoder.classes_)}"
            )

        input_df[column] = encoder.transform([value])

    # Ensure exactly the same feature order as training
    input_df = input_df[X.columns]

    prediction = rf_model.predict(input_df)[0]

    if target_encoder is not None:
        prediction = target_encoder.inverse_transform([prediction])[0]

    return prediction

print("Prediction function created successfully.")


In [ ]:
# CELL 24 — Example Prediction

# This cell automatically creates an example using the first row
# of your dataset, so you do not need to guess the input values.

example_input = {}

for column in X.columns:
    original_column = df[column] if column in df.columns else None

# Use the original dataframe to build a valid example.
example_row = df.drop(columns=[]).iloc[0]

for column in X.columns:
    example_input[column] = example_row[column]

print("Example input:")
print(example_input)

predicted_crop = predict_crop(example_input)

print("\nPredicted Crop:", predicted_crop)


In [ ]:
# CELL 25 — Enter Your Own Values for Prediction

print("Feature columns required by the model:")
print(X.columns.tolist())

user_input = {}

for column in X.columns:
    if column in feature_encoders:
        print(f"\nAvailable values for {column}:")
        print(list(feature_encoders[column].classes_))
        user_input[column] = input(f"Enter {column}: ")
    else:
        user_input[column] = float(input(f"Enter {column}: "))

print("\nInput values:")
print(user_input)

prediction = predict_crop(user_input)

print("\n======================================")
print("       CROP RECOMMENDATION")
print("======================================")
print("Recommended Crop:", prediction)


In [ ]:
# CELL 26 — Save the Trained Model

import joblib

joblib.dump(rf_model, "crop_recommendation_random_forest.pkl")
joblib.dump(feature_encoders, "feature_encoders.pkl")

if target_encoder is not None:
    joblib.dump(target_encoder, "target_encoder.pkl")

print("Model and encoders saved successfully!")


In [ ]:
# CELL 27 — Download the Trained Model Files

from google.colab import files

files.download("crop_recommendation_random_forest.pkl")


In [ ]:
# CELL 28 — Download Feature Encoders

files.download("feature_encoders.pkl")


In [ ]:
# CELL 29 — Download Target Encoder (if created)

if target_encoder is not None:
    files.download("target_encoder.pkl")
else:
    print("Target encoder was not required because the target was already numeric.")
